# SatQuery / DRISHTI - AI Model Worker (Kaggle Host)
### Project: ST-09 SatQuery | Chakravyuh 2026
**Locked System Architecture v1.0**

This notebook runs the VLM inference engine on Kaggle GPU. It connects to Supabase as the data channel and exposes a Gradio public tunnel as the command channel (`share=True`).

### Execution Sequence:
1. **Cell 1**: Run dependency installation (once per session)
2. **Cell 2**: Configure Supabase URL & Service Role Key
3. **Cell 3**: Load Qwen2.5-VL model weights into GPU VRAM (once per session)
4. **Cell 4**: Core functions (inference, structured parsing, tactical drawing)
5. **Cell 5**: Launch Gradio Worker (`share=True`) -> Copy the `https://xxxx.gradio.live` URL and send to Frontend Team

In [ ]:
# CELL 1: Dependency Installation (Run once at start of session)
!pip install -U gradio supabase pillow requests torch torchvision transformers accelerate qwen-vl-utils
# Uninstall incompatible torchaudio to prevent CUDA collision with transformers
!pip uninstall -y torchaudio

In [ ]:
# CELL 2: Supabase Credentials & Configuration
import os
from supabase import create_client, Client

SB_URL = "https://itpfveokpvzcbmoogcui.supabase.co"
# Pre-configured Supabase service_role key for bypassing RLS during AI inference:
try:
    from kaggle_secrets import UserSecretsClient
    SB_KEY = UserSecretsClient().get_secret('SUPABASE_SERVICE_ROLE_KEY')
except Exception:
    SB_KEY = os.environ.get('SUPABASE_SERVICE_ROLE_KEY', 'PASTE_SUPABASE_SERVICE_ROLE_KEY_HERE')

sb: Client = create_client(SB_URL, SB_KEY)
PUB_STORAGE_URL = f"{SB_URL}/storage/v1/object/public/images/"
print("[OK] Supabase connected successfully:", SB_URL)

In [ ]:
# CELL 3: Load VLM Model into GPU VRAM (Run ONCE - never re-run in same session)
import torch
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor

print("Checking GPU availability...")
assert torch.cuda.is_available(), "Please enable GPU accelerator in Kaggle (Settings -> Accelerator -> GPU T4 / P100)"
print(f"GPU Detected: {torch.cuda.get_device_name(0)}")

# Qwen2.5-VL-7B-Instruct (or 3B for lighter memory footprint)
MODEL_ID = "Qwen/Qwen2.5-VL-7B-Instruct"
print(f"Loading {MODEL_ID} into VRAM...")

model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)
processor = AutoProcessor.from_pretrained(MODEL_ID)
print("[OK] Model loaded into VRAM successfully!")

In [ ]:
# CELL 5: The Worker Cell (Launches Gradio Public Tunnel)
import gradio as gr, time

def process(job_id, prompt, image_paths):
    print(f"\n>>> Processing Job: {job_id} | Prompt: {prompt}")
    try:
        if isinstance(image_paths, str):
            image_paths = json.loads(image_paths)
        if not isinstance(image_paths, list):
            image_paths = [image_paths]

        # (1) Download images from Supabase
        imgs = [
            Image.open(requests.get(
                p if p.startswith("http") else PUB_STORAGE_URL + p,
                stream=True, timeout=30
            ).raw).convert("RGB")
            for p in image_paths
        ]

        # (2) Run VLM inference
        raw = ask_model(prompt, imgs)
        answer, boxes = parse_structured(raw)

        # (3) Draw boxes on LAST image (the "After" image)
        marked = draw_tactical_marks(imgs[-1], boxes)

        # (4) Upload marked image to Supabase Storage
        buf = io.BytesIO()
        marked.save(buf, "PNG")
        marked_path = f"marked/{job_id}.png"
        sb.storage.from_("images").upload(
            marked_path, buf.getvalue(),
            file_options={"content-type": "image/png", "upsert": "true"}
        )

        # (5) Update jobs row: status='done'
        sb.table("jobs").update({
            "status": "done",
            "answer": answer,
            "marked_image_path": marked_path
        }).eq("id", job_id).execute()
        print(f">>> Job {job_id} successfully marked and written to Supabase!")
        return answer
    except Exception as e:
        print(f">>> Worker Exception: {e}")
        sb.table("jobs").update({
            "status": "error",
            "answer": f"Worker error: {e}"
        }).eq("id", job_id).execute()
        return f"ERROR: {e}"

custom_css = """
.gradio-container { background-color: #080417 !important; color: #f1f5f9 !important; font-family: sans-serif !important; }
.drishti-header { background: linear-gradient(135deg, #130a2f 0%, #1e1045 100%); border: 1px solid rgba(0, 242, 254, 0.3); border-radius: 16px; padding: 20px; margin-bottom: 20px; }
.drishti-badge { display: inline-block; padding: 4px 10px; background: rgba(0, 242, 254, 0.15); border: 1px solid rgba(0, 242, 254, 0.4); color: #00f2fe; border-radius: 9999px; font-size: 11px; font-weight: 700; text-transform: uppercase; }
.ui-callout { background: rgba(6, 182, 212, 0.1); border-left: 4px solid #00f2fe; padding: 12px 16px; border-radius: 8px; margin-top: 12px; font-size: 13px; color: #e2e8f0; }
"""

with gr.Blocks(title="Drishti / SatQuery AI Worker", css=custom_css) as demo:
    gr.HTML("""
    <div class="drishti-header">
        <span class="drishti-badge">ST-09 Chakravyuh 2026 &bull; Model Host</span>
        <h1 style="color: #ffffff; font-size: 24px; font-weight: 800; margin: 6px 0;">
            DRISHTI <span style="color: #94a3b8; font-weight: 400;">/ SatQuery AI Worker</span>
        </h1>
        <p style="color: #94a3b8; font-size: 13px; margin: 0;">Multimodal Geospatial Intelligence Engine (Qwen2.5-VL) &bull; Model Flags, Humans Decide.</p>
        <div class="ui-callout">
            <strong>Looking for the Drishti Analyst Web UI (Photos 2 & 3)?</strong><br>
            This Gradio page is the <em>backend model worker</em> running on GPU.<br>
            To view and use the official <strong>Drishti Studio interface</strong> with the cosmic sidebar, projects grid, and analysis dropzone, open:
            <a href="http://localhost:3000" target="_blank" style="color: #00f2fe; font-weight: bold; text-decoration: underline;">http://localhost:3000</a>
        </div>
    </div>
    """)
    with gr.Row():
        with gr.Column(scale=3):
            gr.Markdown("### Command Channel Inputs (`/predict`)")
            job_id_input = gr.Textbox(label="job_id (UUID)", value="1a")
            prompt_input = gr.Textbox(label="prompt (Officer Query)", placeholder="e.g. Detect flood inundation extent and embankment breaches.", lines=2)
            image_paths_input = gr.JSON(label="image_paths (Array of Supabase Storage paths)", value=["/samples/amaravati_before.png", "/samples/amaravati_after.png"])
            with gr.Row():
                clear_btn = gr.Button("Clear", variant="secondary")
                submit_btn = gr.Button("Submit Query to Worker", variant="primary")
            gr.Markdown("#### Quick Diagnostic Presets")
            with gr.Row():
                preset_flood = gr.Button("Flood Breach Preset", size="sm")
                preset_bldg = gr.Button("Encroachment Preset", size="sm")
                preset_canopy = gr.Button("Canopy Preset", size="sm")
        with gr.Column(scale=2):
            gr.Markdown("### Inference Status & Output")
            status_output = gr.Textbox(label="Worker Status / Output", lines=8)

    preset_flood.click(fn=lambda: ("demo-flood-" + str(int(time.time())), "Detect flood inundation extent and embankment breaches.", ["/samples/amaravati_before.png", "/samples/amaravati_after.png"]), outputs=[job_id_input, prompt_input, image_paths_input])
    preset_bldg.click(fn=lambda: ("demo-bldg-" + str(int(time.time())), "Count and identify building footprints inside preservation buffer.", ["/mock/marked_sample.png"]), outputs=[job_id_input, prompt_input, image_paths_input])
    preset_canopy.click(fn=lambda: ("demo-canopy-" + str(int(time.time())), "Measure mangrove canopy coverage and foliage density loss.", ["/projects/sundarbans.jpg"]), outputs=[job_id_input, prompt_input, image_paths_input])

    submit_btn.click(fn=process, inputs=[job_id_input, prompt_input, image_paths_input], outputs=status_output, api_name="predict")
    clear_btn.click(fn=lambda: ("", "", []), outputs=[job_id_input, prompt_input, image_paths_input])

demo.launch(share=True)


In [ ]:
# CELL 5: The Worker Cell (Launches Gradio Public Tunnel)
import gradio as gr

def process(job_id, prompt, image_paths):
    print(f"\n>>> Processing Job: {job_id} | Prompt: {prompt}")
    try:
        if isinstance(image_paths, str):
            image_paths = json.loads(image_paths)
        if not isinstance(image_paths, list):
            image_paths = [image_paths]

        # (1) Download images from Supabase
        imgs = [
            Image.open(requests.get(
                p if p.startswith("http") else PUB_STORAGE_URL + p,
                stream=True, timeout=30
            ).raw).convert("RGB")
            for p in image_paths
        ]

        # (2) Run VLM inference
        raw = ask_model(prompt, imgs)
        answer, boxes = parse_structured(raw)

        # (3) Draw boxes on LAST image (the "After" image)
        marked = draw_tactical_marks(imgs[-1], boxes)

        # (4) Upload marked image to Supabase Storage
        buf = io.BytesIO()
        marked.save(buf, "PNG")
        marked_path = f"marked/{job_id}.png"
        sb.storage.from_("images").upload(
            marked_path, buf.getvalue(),
            file_options={"content-type": "image/png", "upsert": "true"}
        )

        # (5) Update jobs row: status='done'
        sb.table("jobs").update({
            "status": "done",
            "answer": answer,
            "marked_image_path": marked_path
        }).eq("id", job_id).execute()
        print(f">>> Job {job_id} successfully marked and written to Supabase!")
        return answer
    except Exception as e:
        print(f">>> Worker Exception: {e}")
        sb.table("jobs").update({
            "status": "error",
            "answer": f"Worker error: {e}"
        }).eq("id", job_id).execute()
        return f"ERROR: {e}"

# Expose Gradio endpoint strictly matching the locked contract
gr.Interface(
    fn=process,
    inputs=[
        gr.Textbox(label="job_id"),
        gr.Textbox(label="prompt"),
        gr.JSON(label="image_paths")
    ],
    outputs=gr.Textbox(label="status"),
    title="SatQuery / Drishti AI Engine"
).launch(share=True)

In [ ]:
# CELL 6 (OPTIONAL RESILIENCE): Background Polling Worker
# Run this if you want the worker to automatically pick up jobs from Supabase
# even if the Gradio tunnel drops (Section 12 variant).
import time

def run_polling_worker():
    print("Starting background Supabase polling listener...")
    while True:
        try:
            res = sb.table("jobs").select("*").eq("status", "processing").limit(1).execute()
            if res.data and len(res.data) > 0:
                job = res.data[0]
                print(f"[POLL] Found processing job: {job['id']}")
                process(job["id"], job["prompt"], job.get("image_paths", []))
            else:
                time.sleep(3)
        except KeyboardInterrupt:
            print("Polling stopped.")
            break
        except Exception as e:
            print(f"Polling error: {e}")
            time.sleep(5)

# Uncomment below to run polling instead of or alongside the tunnel:
# run_polling_worker()